# 11: O Consolidador (relatório final)

Toda análise termina no Consolidador. Antes dele, a resposta final era só a última mensagem do grafo:
às vezes o texto do setor, às vezes o diagnóstico, nunca tudo junto.

O Consolidador **não é um agente com tools**: é uma única chamada ao LLM com saída estruturada
(`RelatorioFinal`), seguida de ajustes em código e da formatação em markdown.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Uma análise completa da VALE3

Esta célula roda o pipeline inteiro (cerca de 30 segundos). A VALE3 costuma acionar o Investigator, então o
relatório terá diagnóstico.

In [2]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro import app

resultado = app.invoke({"messages": [HumanMessage("Analise a ação VALE3")]})
quem_falou = [m.name for m in resultado["messages"] if m.name]
print("Mensagens no histórico:", quem_falou)
print("loop_count:", resultado["loop_count"])

2026-11-06 21:39:11,480 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-06 21:39:12,894 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico de preços, métricas de valuation e balanço patrimonial da VALE3.'


2026-11-06 21:39:12,896 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-06 21:39:12,897 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


2026-11-06 21:39:13,684 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=VALE3.SA


2026-11-06 21:39:13,687 - fluxo_financeiro - INFO - [tool] calcular_metricas_risco: ticker=VALE3.SA


... (32 outputs left out)

2026-11-06 21:39:36,665 - fluxo_financeiro - DEBUG - [router] FINISH → Consolidador


2026-11-06 21:39:36,666 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (9 msgs)


2026-11-06 21:39:38,803 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (2729 chars)


Mensagens no histórico: ['Supervisor', 'YFinance', 'Supervisor', 'Analyzer', 'Supervisor', 'Analyzer', 'Supervisor', 'Investigator', 'Consolidador']
loop_count: 5


## 2. O que o Consolidador recebe e devolve

Ele recebe só o pedido do usuário e as respostas de YFinance, Analyzer e Investigator (as tarefas do
Supervisor ficam de fora), cada uma dentro de `<resposta_agente>`. Devolve um `RelatorioFinal`:

In [3]:
from fluxo_financeiro.agents.consolidator import _insumos

print(_insumos(resultado["messages"])[:1500], "...")

<pedido_usuario>Analise a ação VALE3</pedido_usuario>

<resposta_agente nome="YFinance" n="1">
Dados para VALE3.SA:
Preço Atual: R$ 64.04
P/L:  77.2x
ROE:  0.3%
DY:   6.97%
Industria: Other Industrial Metals & Mining
Sumario de Negócio:
Vale S.A., together with its subsidiaries, produces iron ore and nickel in Brazil, Asia, the Middle East, North Africa, Europe, the Americas, and Oceania. The company operates in two segments, Iron Ore Solutions and Vale Base Metals. It extracts, produces, and distributes iron ore, iron ore pellets, briquettes, nickel, copper, other ferrous products, and by-products, including gold, silver, cobalt, platinum-group metals, and other base metals, as well as low-carbon critical minerals. The company also operates logistics systems and distribution centers, such as mining complexes, railways, and maritime terminals, ports, and ships; generates energy from hydroelectric, solar, and wind sources; and engages in greenfield mineral exploration. In addition, it i

In [4]:
relatorio = resultado["relatorio"]
for campo in ("resumo", "indicadores", "contexto", "riscos", "limitacoes"):
    print(f"--- {campo}:\n{getattr(relatorio, campo)}\n")

--- resumo:
A ação VALE3 fechou cotada a R$ 64,04, refletindo fundamentos pressionados pela queda de 53% no lucro líquido do trimestre (encerrado em 30 de setembro de 2026) na base anual, impulsionada pela retração nos preços globais do minério de ferro e maiores custos operacionais. O perfil da empresa permanece defensivo com Beta de 0,39 e dividend yield atrativo de 6,97%, mas negocia a um P/L elevado de 77,2x e ROE de 0,3%.

--- indicadores:
['Preço Atual: R$ 64,04', 'P/L: 77,2x', 'ROE: 0,3%', 'DY: 6,97%', 'Sharpe Ratio: -0,11', 'Sortino Ratio: -0,18', 'Max Drawdown: -31,5%', 'VaR 95% (diário): -2,91%', 'Beta (vs IBOV): 0,39', 'Alpha (anual): -3,4%', 'Piotroski F-Score: 5/9']

--- contexto:
A Vale atua no segmento de Other Industrial Metals & Mining e enfrenta um ambiente macroeconômico desafiador, marcado pela retração nos preços globais do minério de ferro, menor demanda siderúrgica na China e custos operacionais elevados. Apesar de desafios na margem bruta e no lucro líquido (que

## 3. O diagnóstico é copiado, não resumido

O LLM é instruído a deixar `diagnostico` vazio; o código copia o texto do Investigator literalmente. Assim as
quatro seções (Causa, Natureza, Evidências, Impacto) nunca são reescritas.

In [5]:
texto_investigator = next((m.text for m in resultado["messages"] if m.name == "Investigator"), None)
print("Investigator rodou:", texto_investigator is not None)
print("Diagnóstico idêntico ao texto do Investigator:", relatorio.diagnostico == texto_investigator)

Investigator rodou: True
Diagnóstico idêntico ao texto do Investigator: True


## 4. A mesma função, chamada à parte

`consolidar(messages)` é uma função comum: dá para chamá-la com qualquer histórico. Uma chamada ao LLM.

In [6]:
from fluxo_financeiro.agents.consolidator import consolidar, formatar_relatorio, relatorio_de_emergencia

de_novo = consolidar(resultado["messages"])
print("Indicadores na 2a chamada:", len(de_novo.indicadores), "| na análise:", len(relatorio.indicadores))

Indicadores na 2a chamada: 11 | na análise: 11


## 5. Do relatório estruturado ao markdown

`formatar_relatorio` monta o markdown em código. Seções vazias são omitidas.

In [7]:
from IPython.display import Markdown

Markdown(formatar_relatorio(relatorio, "VALE3"))

# Relatório: VALE3

## Resumo
A ação VALE3 fechou cotada a R$ 64,04, refletindo fundamentos pressionados pela queda de 53% no lucro líquido do trimestre (encerrado em 30 de setembro de 2026) na base anual, impulsionada pela retração nos preços globais do minério de ferro e maiores custos operacionais. O perfil da empresa permanece defensivo com Beta de 0,39 e dividend yield atrativo de 6,97%, mas negocia a um P/L elevado de 77,2x e ROE de 0,3%.

## Indicadores
- Preço Atual: R$ 64,04
- P/L: 77,2x
- ROE: 0,3%
- DY: 6,97%
- Sharpe Ratio: -0,11
- Sortino Ratio: -0,18
- Max Drawdown: -31,5%
- VaR 95% (diário): -2,91%
- Beta (vs IBOV): 0,39
- Alpha (anual): -3,4%
- Piotroski F-Score: 5/9

## Notícias e setor
A Vale atua no segmento de Other Industrial Metals & Mining e enfrenta um ambiente macroeconômico desafiador, marcado pela retração nos preços globais do minério de ferro, menor demanda siderúrgica na China e custos operacionais elevados. Apesar de desafios na margem bruta e no lucro líquido (que caiu para US$ ou R$ 1,25 bilhão no 3T26), a companhia mantém resiliência na geração de caixa operacional e avança nas divisões de cobre e níquel, operando próximo ao piso de sua meta de produção de minério de ferro para 2026.

## Diagnóstico da anomalia
- Causa identificada: A queda de 53% no lucro líquido da VALE3 no 3T26 (comparado ao 3T25) foi impulsionada primariamente pela queda nos preços realizados de minério de ferro (reflexo da menor demanda siderúrgica na China), associada à pressão de custos operacionais (maiores despesas com frete, combustíveis e impactos cambiais pela variação do BRL frente ao dólar).
- Natureza do evento: Cíclico / Setorial (atrelado diretamente ao ciclo de commodities, à volatilidade dos preços internacionais do minério de ferro e à dinâmica macroeconômica chinesa).
- Evidências: Divulgação do resultado do 3T26 da companhia, relatórios corporativos da Vale e notícias de mercado indexadas nos fatos relevantes.
- Impacto no investimento: O evento reflete a forte exposição da mineradora aos ciclos de preços globais e à desaceleração imobiliária/industrial da China. Não se trata de uma falha estrutural de operação da empresa (visto que o volume e a divisão de metais básicos mantiveram resiliência), mas impõe cautela quanto à volatilidade das margens e à geração de caixa para dividendos em momentos de baixa do minério.

## Riscos
- Forte exposição aos ciclos de preços globais de commodities e à desaceleração industrial e imobiliária da China
- Pressão de custos operacionais, despesas com frete, combustíveis e impactos cambiais
- Possível limitação no pagamento de dividendos extraordinários no curto prazo
- Volatilidade elevada e retração nas margens operacionais


## 6. Se o LLM do Consolidador falhar

A análise já custou várias chamadas; perder tudo no último passo seria ruim. Se `consolidar` levantar erro
(cota, rede, saída inválida), o nó usa `relatorio_de_emergencia`: as respostas dos agentes juntadas, sem LLM,
e `relatorio = None` no estado.

In [8]:
print(relatorio_de_emergencia(resultado["messages"])[:800], "...")

> Não foi possível consolidar o relatório; seguem as respostas dos agentes.

## YFinance
Dados para VALE3.SA:
Preço Atual: R$ 64.04
P/L:  77.2x
ROE:  0.3%
DY:   6.97%
Industria: Other Industrial Metals & Mining
Sumario de Negócio:
Vale S.A., together with its subsidiaries, produces iron ore and nickel in Brazil, Asia, the Middle East, North Africa, Europe, the Americas, and Oceania. The company operates in two segments, Iron Ore Solutions and Vale Base Metals. It extracts, produces, and distributes iron ore, iron ore pellets, briquettes, nickel, copper, other ferrous products, and by-products, including gold, silver, cobalt, platinum-group metals, and other base metals, as well as low-carbon critical minerals. The company also operates logistics systems and distribution centers, such as mi ...


## Sem recomendação de compra ou venda

O relatório descreve o que os dados mostram e não dá veredito nem preço-alvo. Um "compre" escrito por um LLM a
partir de poucas fontes passaria uma confiança que os dados não sustentam.

## O que observar

- `limitacoes`: lista o que faltou ou falhou (busca sem resultado, etapa que não rodou). Parte delas é escrita em código, não depende de o LLM perceber.
- Os números de `indicadores` precisam aparecer nas respostas dos agentes: o prompt proíbe inventar ou recalcular.
- A segunda chamada a `consolidar` pode escrever frases diferentes da primeira (o LLM não é determinístico), mas os números são os mesmos, porque vêm dos insumos.
- O relatório de emergência é feio, mas não perde nada do que os agentes produziram.